# Predicting Celiac Disease Diagnosis

Motivating Problem:


Imports:

In [4]:
import pandas as pd
import numpy as np
from pandas import read_csv
from pandas import DataFrame as df
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

**Data:**
I am using open source data from Kaggle found [here](https://www.kaggle.com/code/minaremon39/celiac-disease-prediction/notebook)

In [5]:
# First, reading in the data from the CSV file
# if you are trying to run this code, make sure you have downloaded the CSV file

filename = "celiac_disease_lab_data.csv"
df = pd.read_csv(filename)
# printing the first 5 rows to make sure the data was read in correctly
df.head()

,Age,Gender,Diabetes,Diabetes_Type,Diarrhea,Abdominal,Short_Stature,Sticky_Stool,Weight_loss,IgA,IgG,IgM,Marsh,cd_type,Disease_Diagnose
0,10,Male,Yes,Type 1,inflammatory,yes,PSS,no,no,1.30,10.0,1.00,marsh type 0,potential,yes
1,9,Male,Yes,Type 1,fatty,yes,PSS,no,no,1.50,12.5,1.30,marsh type 3a,atypical,yes
2,8,Female,Yes,Type 1,watery,yes,Variant,yes,yes,0.40,8.0,0.50,marsh type 1,latent,yes
3,10,Male,Yes,Type 1,watery,yes,PSS,no,no,0.98,9.0,0.66,marsh type 3a,silent,yes
4,9,Male,Yes,Type 1,fatty,yes,PSS,no,no,1.00,10.5,1.10,marsh type 1,latent,yes


This has loaded properly, so we will move on to setting up for machine learning. Next, we will look at the datatypes of each variable and look for missing values.

In [6]:
df.isnull().sum()

Age                   0
Gender                0
Diabetes              0
Diabetes_Type       418
Diarrhea              0
Abdominal             0
Short_Stature         0
Sticky_Stool          0
Weight_loss           0
IgA                   0
IgG                   0
IgM                   0
Marsh                 0
cd_type               0
Disease_Diagnose      0
dtype: int64

This means there's 418 missing values in the Diabetes_Type variable that we will have to handle. There aren't any values missing from the .csv file, so something must have happened when reading the data in. We will replace these NA values with 0s since they represent people without diabetes.

In [7]:
df["Diabetes_Type"] = df["Diabetes_Type"].fillna(0)

In [8]:
df.isnull().sum()

Age                 0
Gender              0
Diabetes            0
Diabetes_Type       0
Diarrhea            0
Abdominal           0
Short_Stature       0
Sticky_Stool        0
Weight_loss         0
IgA                 0
IgG                 0
IgM                 0
Marsh               0
cd_type             0
Disease_Diagnose    0
dtype: int64

Now we have no remaining missing values! Let's move on to our types of variables.

In [9]:
df.dtypes

Age                   int64
Gender               object
Diabetes             object
Diabetes_Type        object
Diarrhea             object
Abdominal            object
Short_Stature        object
Sticky_Stool         object
Weight_loss          object
IgA                 float64
IgG                 float64
IgM                 float64
Marsh                object
cd_type              object
Disease_Diagnose     object
dtype: object

Since only 4 of our variables are numeric, we will have to one-hot encode the rest of the 'object' variables. This will create several binary variables for each categorical variable. The 'Diabetes Type' variable has three levels, Type I, Type II, and None. Three binary variables need to be created for this, one binary indicator for Type I, one for Type II, and one for none. We are going to use pandas to do this automatically. Before we can start, we first need to convert some of the variables with only two levels to binary variables. 

The variables that need to be converted to binary are all of the yes/no variables in the dataframe.

Creating one binary variable instead of one-hot encoding and making multiple binary variables is more efficient and will make modeling just a little bit easier. 

This will apply to the 'Diabetes', 'Abdominal', 'Sticky_Stool', 'Weight_loss', and 'Disease_Diagnose' variables.

First, notice that in the original dataset, some of the answers are Yes with a capital Y and some are lowercase. We'll convert all of the y/n variabes to lowercase to prevent any case-sensitivity issues.

In [10]:
df['Diabetes'] = df['Diabetes'].str.lower()
df['Abdominal'] = df['Abdominal'].str.lower()
df['Sticky_Stool'] = df['Sticky_Stool'].str.lower()
df['Weight_loss'] = df['Weight_loss'].str.lower()
df['Disease_Diagnose'] = df['Disease_Diagnose'].str.lower()

Now we should have all lowercase yes and no values in these variables!

In [11]:
df.head()

,Age,Gender,Diabetes,Diabetes_Type,Diarrhea,Abdominal,Short_Stature,Sticky_Stool,Weight_loss,IgA,IgG,IgM,Marsh,cd_type,Disease_Diagnose
0,10,Male,yes,Type 1,inflammatory,yes,PSS,no,no,1.30,10.0,1.00,marsh type 0,potential,yes
1,9,Male,yes,Type 1,fatty,yes,PSS,no,no,1.50,12.5,1.30,marsh type 3a,atypical,yes
2,8,Female,yes,Type 1,watery,yes,Variant,yes,yes,0.40,8.0,0.50,marsh type 1,latent,yes
3,10,Male,yes,Type 1,watery,yes,PSS,no,no,0.98,9.0,0.66,marsh type 3a,silent,yes
4,9,Male,yes,Type 1,fatty,yes,PSS,no,no,1.00,10.5,1.10,marsh type 1,latent,yes


In [12]:
# Let's check how many yes/no values we have in the Diabetes column
df['Diabetes'].value_counts()

Diabetes
yes    1829
no      377
Name: count, dtype: int64

In [13]:
# overwrite the 'Diabetes' col with 1s for yes and 0s otherwise
df['Diabetes'] = np.where(df['Diabetes'] == 'yes', 1, 0)
df.head()

,Age,Gender,Diabetes,Diabetes_Type,Diarrhea,Abdominal,Short_Stature,Sticky_Stool,Weight_loss,IgA,IgG,IgM,Marsh,cd_type,Disease_Diagnose
0,10,Male,1,Type 1,inflammatory,yes,PSS,no,no,1.30,10.0,1.00,marsh type 0,potential,yes
1,9,Male,1,Type 1,fatty,yes,PSS,no,no,1.50,12.5,1.30,marsh type 3a,atypical,yes
2,8,Female,1,Type 1,watery,yes,Variant,yes,yes,0.40,8.0,0.50,marsh type 1,latent,yes
3,10,Male,1,Type 1,watery,yes,PSS,no,no,0.98,9.0,0.66,marsh type 3a,silent,yes
4,9,Male,1,Type 1,fatty,yes,PSS,no,no,1.00,10.5,1.10,marsh type 1,latent,yes


Let's check how many 1/0 values we have in the Diabetes column

In [14]:
# Here's what we had before:
# Diabetes
# yes    1829
# no      377
# Name: count, dtype: int64
df['Diabetes'].value_counts()

Diabetes
1    1829
0     377
Name: count, dtype: int64

Our counts line up, so we've correctly transformed this variable! Let's do the same for the remaining binary variables. 

In [15]:
# First, we will get the initial value counts so we can check our work
df['Abdominal'].value_counts()

Abdominal
yes    1781
no      425
Name: count, dtype: int64

In [16]:
# Next, we will convert the 'Abdominal' column to 1s and 0s
df['Abdominal'] = np.where(df['Abdominal'] == 'yes', 1, 0)

In [17]:
# Finally, we will check our work
df['Abdominal'].value_counts()

Abdominal
1    1781
0     425
Name: count, dtype: int64

In [18]:
df['Sticky_Stool'].value_counts()

Sticky_Stool
yes    1820
no      386
Name: count, dtype: int64

In [19]:
df["Sticky_Stool"] = np.where(df["Sticky_Stool"] == 'yes', 1, 0)
df["Sticky_Stool"].value_counts()

Sticky_Stool
1    1820
0     386
Name: count, dtype: int64

In [20]:
df['Weight_loss'].value_counts()

Weight_loss
yes    1514
no      692
Name: count, dtype: int64

In [21]:
df["Weight_loss"] = np.where(df["Weight_loss"] == 'yes', 1, 0)
df["Weight_loss"].value_counts()

Weight_loss
1    1514
0     692
Name: count, dtype: int64

In [22]:
df['Disease_Diagnose'].value_counts()

Disease_Diagnose
yes    1843
no      363
Name: count, dtype: int64

In [23]:
df["Disease_Diagnose"] = np.where(df["Disease_Diagnose"] == 'yes', 1, 0)
df["Disease_Diagnose"].value_counts()

Disease_Diagnose
1    1843
0     363
Name: count, dtype: int64

In [24]:
df.head()

,Age,Gender,Diabetes,Diabetes_Type,Diarrhea,Abdominal,Short_Stature,Sticky_Stool,Weight_loss,IgA,IgG,IgM,Marsh,cd_type,Disease_Diagnose
0,10,Male,1,Type 1,inflammatory,1,PSS,0,0,1.30,10.0,1.00,marsh type 0,potential,1
1,9,Male,1,Type 1,fatty,1,PSS,0,0,1.50,12.5,1.30,marsh type 3a,atypical,1
2,8,Female,1,Type 1,watery,1,Variant,1,1,0.40,8.0,0.50,marsh type 1,latent,1
3,10,Male,1,Type 1,watery,1,PSS,0,0,0.98,9.0,0.66,marsh type 3a,silent,1
4,9,Male,1,Type 1,fatty,1,PSS,0,0,1.00,10.5,1.10,marsh type 1,latent,1


Since there are only two options for Gender, we will code Male as 1 and Female as 0

In [25]:
df['Gender'] = np.where(df['Gender'] == 'Male', 1, 0)
df.head()

,Age,Gender,Diabetes,Diabetes_Type,Diarrhea,Abdominal,Short_Stature,Sticky_Stool,Weight_loss,IgA,IgG,IgM,Marsh,cd_type,Disease_Diagnose
0,10,1,1,Type 1,inflammatory,1,PSS,0,0,1.30,10.0,1.00,marsh type 0,potential,1
1,9,1,1,Type 1,fatty,1,PSS,0,0,1.50,12.5,1.30,marsh type 3a,atypical,1
2,8,0,1,Type 1,watery,1,Variant,1,1,0.40,8.0,0.50,marsh type 1,latent,1
3,10,1,1,Type 1,watery,1,PSS,0,0,0.98,9.0,0.66,marsh type 3a,silent,1
4,9,1,1,Type 1,fatty,1,PSS,0,0,1.00,10.5,1.10,marsh type 1,latent,1


Now, we should recheck our datatypes to determine which variables we have to one-hot encode.

In [26]:
df.dtypes

Age                   int64
Gender                int32
Diabetes              int32
Diabetes_Type        object
Diarrhea             object
Abdominal             int32
Short_Stature        object
Sticky_Stool          int32
Weight_loss           int32
IgA                 float64
IgG                 float64
IgM                 float64
Marsh                object
cd_type              object
Disease_Diagnose      int32
dtype: object

Time to one-hot encode all of our categorical features! We will use one of the pandas methods, get_dummies. This will only apply to the object datatypes, so we don't need to repeat it for every variable we want to encode. I am going to overwrite our original dataframe with the one-hot encoded variables.

In [27]:
df = pd.get_dummies(df, columns=['Diabetes_Type', ], prefix='DiabetesType')

In [28]:
# Since the label for each column auto filled, we're going to change DiabetesType_0 to Diabetes_Type_None
df = df.rename(columns={'DiabetesType_0': 'Diabetes_Type_None'})
#df.head()
# much better :)

Since we know (based on variable type) which variables we need one-hot encode, we can do the rest of them in one command rather than doing each individually!

In [29]:
df = pd.get_dummies(df,drop_first=True)

Let's look at our new dataframe

In [30]:
df.head()

,Age,Gender,Diabetes,Abdominal,Sticky_Stool,Weight_loss,IgA,IgG,IgM,Disease_Diagnose,...,Marsh_marsh type 2,Marsh_marsh type 3a,Marsh_marsh type 3b,Marsh_marsh type 3c,Marsh_none,cd_type_latent,cd_type_none,cd_type_potential,cd_type_silent,cd_type_typical
0,10,1,1,1,0,0,1.30,10.0,1.00,1,...,False,False,False,False,False,False,False,True,False,False
1,9,1,1,1,0,0,1.50,12.5,1.30,1,...,False,True,False,False,False,False,False,False,False,False
2,8,0,1,1,1,1,0.40,8.0,0.50,1,...,False,False,False,False,False,True,False,False,False,False
3,10,1,1,1,0,0,0.98,9.0,0.66,1,...,False,True,False,False,False,False,False,False,True,False
4,9,1,1,1,0,0,1.00,10.5,1.10,1,...,False,False,False,False,False,True,False,False,False,False


In [31]:
df.dtypes

Age                         int64
Gender                      int32
Diabetes                    int32
Abdominal                   int32
Sticky_Stool                int32
Weight_loss                 int32
IgA                       float64
IgG                       float64
IgM                       float64
Disease_Diagnose            int32
Diabetes_Type_None           bool
DiabetesType_Type 1          bool
DiabetesType_Type 2          bool
Diarrhea_inflammatory        bool
Diarrhea_watery              bool
Short_Stature_PSS            bool
Short_Stature_Variant        bool
Marsh_marsh type 1           bool
Marsh_marsh type 2           bool
Marsh_marsh type 3a          bool
Marsh_marsh type 3b          bool
Marsh_marsh type 3c          bool
Marsh_none                   bool
cd_type_latent               bool
cd_type_none                 bool
cd_type_potential            bool
cd_type_silent               bool
cd_type_typical              bool
dtype: object

Now every variable is numeric or a boolean so we can move on!

# We are almost able to build a machine learning model with this data! 
First, we will separate the label, the disease_diagnosis column, from the rest of the features in order to prevent the models from accessing the label data. Next, we will split the data into a training and a test set. We will use 20% of the data for our test set and the rest for our training set. 

In [32]:
y = df['Disease_Diagnose']
x = df.drop('Disease_Diagnose', axis=1)

Let's discuss some of the parameters in the next fuction call. First, we are using x as our data and y as our label (what we're trying to predict). The test_size parameter is where we section out 0.2 of the data for our test set. The random_state parameter is setting a seed so the same split of the data will be done every call. This will later allow us to directly compare models trained on the same split of data.

In [33]:
X_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [34]:
# Let's look at the shape of each of these new datasets
print(X_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(1764, 27)
(442, 27)
(1764,)
(442,)


# We are ready to build our first model!
We will start with a simple logistic regression model. I am going to import the library we need here, so it is more clear what library we are using.

In [35]:
from sklearn.linear_model import LogisticRegression

In [36]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Now we will test the model on our test set.

In [41]:
predictions = model.predict(x_test)

In [43]:
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error

In [44]:
mae = mean_absolute_error(y_test, predictions)

In [45]:
print("Mean Absolute Error:", mae)

Mean Absolute Error: 0.0022624434389140274
